# Tutorial 11: DeepSeek Sparse Attention

This notebook is intentionally thin. The interesting logic lives in `triton_tutorials/11_deepseek_sparse_attention/dsa_runtime.py`, while the notebook stays responsible for the tutorial-style namespace and execution flow used by the Modal wrappers.

## Optimization Ladder

1. Match the contest reference exactly.
2. Isolate the expensive loops.
3. Specialize only those loops in Triton.
4. Keep selection / orchestration in PyTorch when Triton would add complexity without enough payoff.

That is why the indexer keeps `torch.topk` on the host side while the score accumulation and sparse attention paths are Triton-backed.

In [ ]:
from pathlib import Path
import sys

for candidate in [
    Path('/root/triton_tutorials/11_deepseek_sparse_attention'),
    Path.cwd() / 'triton_tutorials' / '11_deepseek_sparse_attention',
    Path.cwd().parent / '11_deepseek_sparse_attention',
]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

from dsa_runtime import make_runtime_namespace

_runtime_ns = make_runtime_namespace()
globals().update(vars(_runtime_ns))


## Suggested walkthrough

- Start with `make_lightning_indexer_case()` and `lightning_indexer_reference()`.
- Inspect `dequant_fp8_kv_cache_torch()` before touching the Triton score kernel.
- Then compare `sparse_mla_attention_reference()` with `sparse_mla_attention()` to see why the implementation uses a two-pass LSE/output split to reduce register pressure for the 512-wide MLA value path.
